# Rows Without An Answer

**Version**: 1

**Last modified**: 2026-10-12

**Compatible with opengate-gate-tree version:** 0.5.0

**Author**: Mateusz Jakub Bała

## About

Some rows have no answer to give. A vector of no length has no direction, two
parallel directions span no plane, and a number GATE wrote may mean nothing this
version of the package knows. The package answers such a row with `nan` (or
`None`), counts how many there were, and says so in the log — and carries on
with the rest of the column.

That is a choice, and this notebook shows both sides of it: why one degenerate
row out of half a million must not end an analysis, and why a column that
quietly holds `nan` looks, to anyone who has not turned the log on, like a
library that loses data.

## What will you learn from this notebook

- which calculations answer `nan` for a row, and when
- how to turn the log on, so that the package says how many rows it could not
  answer
- how to count those rows yourself
- where the rule stops: directions that are parallel only up to the precision
  of the file get an answer, and deciding which of them mean nothing is your
  call
- how a value the package does not know is read, and how a statistics report
  counts what is not a number

## Tutorial

Every step below is a line of text, the code that does what the line says, and
a word about the answer where the answer needs one.

In [1]:
from tools import environment, paths

print(environment.describe())

opengate-gate-tree: 0.5.0
python: 3.14.5
run on: 2026-10-12


The header of this notebook states the version it was written for; the line
above states the version it just ran against. If they differ, the notebook may
still be right — but nobody has checked it against what is installed.

Two small GATE outputs carried in the repository, both described in
[`../data/README.md`](../data/README.md): a "Hits" tree with no system, and
one written with a `PositroniumSource`.

In [2]:
output = paths.output_dir("tutorial-16-rows-without-an-answer")
example_input = paths.input_example("hits-a1-no-system.root")
positronium_input = paths.input_example("positronium-source-all-variants.root")

In [3]:
import logging

import numpy as np
import pandas as pd

from opengate_gate_tree import (
    GateTree,
    TreeData,
    angle_between,
    compute_statistics,
    decode_positronium_column,
    format_statistics,
    momentum_direction_from_positions,
    normalize,
    plane_normal,
    read_tree,
)
from opengate_gate_tree.logging_setup import configure_logging

frame = read_tree(example_input, GateTree.HITS).to_dataframe()

### A Whole Column Without An Answer

Every gamma in this file was born at the origin:

In [4]:
source = frame.gate.source_position()
source.to_frame().drop_duplicates()

,sourcePosX,sourcePosY,sourcePosZ
0,0.0,0.0,0.0


A position at the origin has a radius of zero and no angles at all. Asking for
its spherical components answers the radius and leaves both angles empty:

In [5]:
spherical = source.spherical()
spherical.head(3)

,radius,polar,azimuth
0,0.0,NaN,NaN
1,0.0,NaN,NaN
2,0.0,NaN,NaN


In [6]:
spherical.isna().sum()

radius       0
polar      500
azimuth    500
dtype: int64

Five hundred rows out of five hundred have no angles, and the cell above said
nothing about it. That is on purpose: importing the package does not configure
logging, so a library used inside somebody else's application stays quiet
until that application asks otherwise. The messages are written all the same —
they go nowhere.

### Turning The Log On

`configure_logging` gives the package logger somewhere to write. It can be
called again, and changes only the level then, so the cell can be run twice
without printing everything twice. Warnings are the level these reports are
written at:

In [7]:
configure_logging(logging.WARNING)

spherical = source.spherical()

2026-10-12 03:24:41,139 [WARNING] opengate_gate_tree: 500 of 500 position values have no length to divide out, so they have no direction. Those rows are read as nothing at all.


The same call, and now the package says what it did: how many rows, out of
how many, and what it made of them. The message goes to the standard error
stream, with a time stamp and the name of the logger, which is how the command
line shows it too.

### A Few Rows Without An Answer

A real file rarely fails a whole column. Three directions, the middle one of no
length — written by hand, since the example file has no such row:

In [8]:
directions = np.array(
    [
        [3.0, 4.0, 0.0],
        [0.0, 0.0, 0.0],
        [0.0, 0.0, 2.0],
    ]
)

normalize(directions)

2026-10-12 03:24:41,154 [WARNING] opengate_gate_tree: 1 of 3 vector values have no length to divide out, so they have no direction. Those rows are read as nothing at all.


array([[0.6, 0.8, 0. ],
       [nan, nan, nan],
       [0. , 0. , 1. ]])

The first and the last row are answered, the middle one is `nan`, and the
log names one row of three. Everything built on top of a direction follows the
same rule — an angle, a plane, a rebuilt direction of flight:

In [9]:
angle_between(directions, [[1.0, 0.0, 0.0]] * 3)

2026-10-12 03:24:41,169 [WARNING] opengate_gate_tree: 1 of 3 vector values have no length to divide out, so they have no direction. Those rows are read as nothing at all.


array([0.92729522,        nan, 1.57079633])

Refusing the whole column would be the other choice, and the wrong one: an
analysis of half a million hits must not stop at the one hit that happened to
sit at the origin. The price is that a `nan` can travel a long way before
anybody looks at it, which is why the log says where it started.

### Counting Them Yourself

The log is for a person. Code counts the rows itself — a row of vectors is
unanswered when any of its components is `nan`, and a column of numbers is
asked with `isna()`:

In [10]:
{
    "vectors without a direction": int(np.isnan(normalize(directions)).any(axis=1).sum()),
    "positions without a polar angle": int(spherical["polar"].isna().sum()),
}

2026-10-12 03:24:41,176 [WARNING] opengate_gate_tree: 1 of 3 vector values have no length to divide out, so they have no direction. Those rows are read as nothing at all.


{'vectors without a direction': 1, 'positions without a polar angle': 500}

### Where The Rule Stops

A row is answered with `nan` when the length that has to be divided out is
below 10⁻¹²: then there is no direction left to speak of. Directions read from
a GATE file are rarely that exact, and this is where the analysis has to take
over.

The first hit of every track: where the gamma arrived from, rebuilt from the
source and the hit, and the direction GATE wrote at the hit, which is where
the gamma went next.

In [11]:
TRACK_KEY = ["runID", "eventID", "trackID"]

first = frame[frame.groupby(TRACK_KEY).cumcount() == 0]

arrived = momentum_direction_from_positions(
    first[["sourcePosX", "sourcePosY", "sourcePosZ"]].to_numpy(),
    first[["posX", "posY", "posZ"]].to_numpy(),
)
left = first[["momDirX", "momDirY", "momDirZ"]].to_numpy()

turn = pd.Series(angle_between(arrived, left), index=first.index, name="turn")
turn.groupby(first["processName"]).agg(["count", "min", "max"])

,count,min,max
processName,,,
Compton,151,8.962724e-02,3.041015e+00
PhotoElectric,149,4.091839e-10,5.032330e-08


A Compton scattering turned its gamma by at least 0.09 rad, about 5°. A
photoelectric absorption ends the gamma, and the direction GATE writes there is
the one it arrived with — up to a few times 10⁻⁸ rad, the precision of the
`float32` positions the arrival was rebuilt from.

The plane through the two directions is what a polarization is estimated from.
For an absorption there is no such plane, but the package cannot know that:
the directions are not parallel to within 10⁻¹², so every row gets a normal,
and the log says nothing.

In [12]:
normals = plane_normal(arrived, left)

{
    "rows": len(normals),
    "rows answered with nan": int(np.isnan(normals).any(axis=1).sum()),
}

{'rows': 300, 'rows answered with nan': 0}

A hundred and forty-nine of those normals are rounding noise with a length of
one. Which angle is too small to define a plane is a question about the data,
not about arithmetic, so the analysis answers it — here at 10⁻⁶ rad, far below
the smallest real scattering and far above the noise — and empties those rows
itself:

In [13]:
NO_PLANE_BELOW = 1e-6  # rad

no_plane = turn.to_numpy() < NO_PLANE_BELOW
normals[no_plane] = np.nan

{
    "rows without a plane": int(no_plane.sum()),
    "photoelectric absorptions": int((first["processName"] == "PhotoElectric").sum()),
}

{'rows without a plane': 149, 'photoelectric absorptions': 149}

The rows emptied are exactly the absorptions. The same reasoning applies to
any direction rebuilt from two nearby hits: the package answers whatever is
not exactly degenerate, and what counts as degenerate for a measurement is the
measurement's business.

### A Value Nobody Described

The branches a `PositroniumSource` writes hold numbers with a meaning, and
`decode_positronium_column` reads them as names. On the example file every
value is known, so the log stays quiet:

In [14]:
gamma_types = read_tree(positronium_input, GateTree.HITS, ["gammaType"])["gammaType"]

pd.Series(decode_positronium_column("gammaType", gamma_types)).value_counts()

2    367
3     72
0     61
Name: count, dtype: int64

A GATE build may write a value this version does not describe. Such a value
is read as `None` — not as a guess — and reported with how often it occurs.
The column below is the file's column with one value added by hand, 7, which
no `GammaType` stands for:

In [15]:
with_unknown = np.append(gamma_types, 7)

decoded = decode_positronium_column("gammaType", with_unknown)
decoded[-3:]

2026-10-12 03:24:41,242 [WARNING] opengate_gate_tree: Branch 'gammaType' holds 1 value(s) this version does not describe: 7 (1 row(s)). They are read as nothing at all.


array([<GammaType.ANNIHILATION: 2>, <GammaType.ANNIHILATION: 2>, None],
      dtype=object)

In [16]:
int(pd.isna(decoded).sum())

1

`pd.isna` counts `None` the same way it counts `nan`, so one line covers both
kinds of row without an answer.

### What A Report Says About Them

A statistics report counts values that are not finite rather than averaging
them, because a single `nan` would make the mean and the range `nan` too, and
JSON could not even write it. The z component of the plane normals, with the
absorptions emptied, next to the identifiers of the rows:

In [17]:
with_normals = first[TRACK_KEY].assign(normalZ=normals[:, 2])

report = compute_statistics(TreeData.from_dataframe(GateTree.HITS, with_normals))
print(format_statistics(report))

Tree: Hits
Entries: 300
Events: 150 (counted by runID, eventID)
Runs: 1
Tracks: 300 (counted by runID, eventID, trackID)
Branches (4):
  - runID / int32: min 0, max 0, mean 0, std 0, 1 distinct
  - eventID / int32: min 0, max 149, mean 74.5, std 43.3003, 150 distinct
  - trackID / int32: min 1, max 2, mean 1.5, std 0.5, 2 distinct
  - normalZ / float64: min -0.97009, max 0.96866, mean 0.0161006, std 0.569464, not finite 149


In [18]:
next(branch for branch in report.branches if branch.name == "normalZ")

BranchStatistics(name='normalZ', dtype='float64', kind=<BranchKind.FLOAT: 'float'>, entries=300, non_finite_count=149, minimum=-0.9700900909607181, maximum=0.9686596236961666, mean=0.016100561122593025, std=0.5694638624351741, unique_count=None, top_values=())

The range and the mean are those of the 151 rows that have a plane, and
`non_finite_count` says how many rows were left out of them — the same number
the analysis emptied by hand. The JSON file `write_statistics` writes calls the
same number `not_finite`.

## Summary

A row the package cannot answer comes back as `nan` or `None`, the column
carries on, and the log — once `configure_logging` gives it somewhere to
write — says how many rows there were. Counting them is one `isna()` away;
deciding that a row which *was* answered means nothing, as with directions
parallel up to the precision of the file, is left to the analysis.

## Bibliography

- [Vectors and angles](https://opengate-gate-tree.readthedocs.io/en/latest/guide/vectors.html)
  — rows that answer nothing, and the precision of directions read from a file
- [PositroniumSource Data](https://opengate-gate-tree.readthedocs.io/en/latest/guide/positronium.html)
  — reading values as names, and a value the package does not describe
- [Statistics](https://opengate-gate-tree.readthedocs.io/en/latest/guide/statistics.html)
  — how a report counts values that are not finite
- [Using the package as a library](https://opengate-gate-tree.readthedocs.io/en/latest/guide/library.html)
  — logging, and why the package is quiet until asked

## Changelog

- v1: written.